In [10]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# read all the pdf file in directory

def process_all_pdfs(pdf_directory):
	all_docs=[]
	pdf_dir = Path(pdf_directory)

	# find all pdf
	files = list(pdf_dir.glob('**/*.pdf'))
	print(f"found {len(files)} files")

	for file in files:
		print(f"\nProcessing: {file.name}")
		try:
			loader = PyPDFLoader(str(file))
			docs =loader.load()

			for doc in docs:
				doc.metadata['source_file'] = file.name
				doc.metadata['file_type'] = 'pdf'

			all_docs.extend(docs)
			print(f"loaded {len(docs)} pages")
		except Exception as e:
			print(f"Error: {e}")
	
	print(f"\nTotal document loaded: {len(all_docs)}")
	
	return all_docs

In [15]:
all_pdf_docs = process_all_pdfs("../data/pdf2")

found 4 files

Processing: huahin.pdf
loaded 44 pages

Processing: bkk.pdf
loaded 44 pages

Processing: phuket.pdf
loaded 28 pages

Processing: chiangmai.pdf
loaded 52 pages

Total document loaded: 168


In [18]:
# splitting the text

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
	text_splitter = RecursiveCharacterTextSplitter(
		chunk_size=chunk_size,
		chunk_overlap=chunk_overlap,
		length_function=len,
		separators=["\n\n", "\n", " ", ""]
	)
	split_docs = text_splitter.split_documents(documents)
	print(f"split{len(documents)} documents into {len(split_docs)} chunks")

	# show sample of chunks

	if split_docs:
		print(f"\nExample chunk:")
		print(f"\nContent:{split_docs[0].page_content[:200]}...")
		print(f"\nMeata: {split_docs[0].metadata}")
	return split_docs

In [19]:
chunks = split_documents(all_pdf_docs)

split168 documents into 350 chunks

Example chunk:

Content:Cover_m14.indd   1 3/4/20   21:16...

Meata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'creationdate': '2020-03-04T21:15:58+07:00', 'gts_pdfxversion': 'PDF/X-4', 'moddate': '2020-08-24T12:58:17+02:00', 'title': 'Cover_m14.indd', 'trapped': '/False', 'source': '../data/pdf2/huahin.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': 'huahin.pdf', 'file_type': 'pdf'}


In [20]:
# embed the text
texts = [doc.page_content for doc in chunks]
texts

['Cover_m14.indd   1 3/4/20   21:16',
 'Hua Hin Beach\n2-43_m14.indd   2 3/24/20   11:28',
 'CONTENTS\nHUA HIN 8\n City Attractions 9\n Activities 15\n How to Get There 16\n Special Event 16\nPRACHUAP KHIRI KHAN 18\n City Attractions 19\n Out-Of-City Attractions 19\n Local Products 23\n How to Get There 23\nCHA-AM 24\n Attractions 25\n How to Get There 25\nPHETCHABURI 28\n City Attractions 29\n Out-Of-City Attractions 32\n Special Events 34\n Local Products 35\n How to Get There 35\nRATCHABURI 36\n City Attractions 37\n Out-Of-City Attractions 37\n Local Products 43\n How to Get There 43\n2-43_m14.indd   3 3/24/20   11:28',
 'HUA HIN & CHA-AM\nHUA HIN & CHA-AM\n Prachuap Khiri Khan     Phetchaburi    Ratchaburi\n2-43_m14.indd   4 3/24/20   11:28',
 '2-43_m14.indd   5 3/24/20   11:28',
 'The Republic of the \nUnion of Myanmar\nThe Kingdom\nof Cambodia\n2-43_m14.indd   6 3/24/20   11:28',
 'The Republic of the \nUnion of Myanmar\nThe Kingdom\nof Cambodia\n2-43_m14.indd   7 3/24/20   11:2

In [21]:
class EmbeddingManager:
	
	def __init__(self, model_name:str='all-MiniLM-L6-v2'):
		self.model_name = model_name
		self.model = None
		self.load_model()

	def load_model(self):
		try:
			print(f"loading embedding model: {self.model_name}")
			self.model = SentenceTransformer(self.model_name)
			print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
		except Exception as e:
			print(f"Error loading model {self.model_name}: {e}")
			raise
	
	def generate_embdeddings(self, texts: List[str]) -> np.ndarray:
		if not self.model:
			raise ValueError("Model not loaded")
		
		print(f"Generating embeddings for {len(texts)} texts...")
		embeddings = self.model.encode(texts, show_progress_bar=True)
		print(f"Genrerated embeddings with shape: {embeddings.shape}")
		return embeddings

In [22]:
embedding_manager = EmbeddingManager()

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7536.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [31]:
# class for vectordb

class VectorStore:
	
	def __init__(self, collection_name:str="pdf_documents", persist_directory:str="../data/vector_store2"):
		self.collection_name = collection_name
		self.persist_directory = persist_directory
		self.client = None
		self.collection = None
		self._initialize_store()

	def _initialize_store(self):
		try:
			os.makedirs(self.persist_directory, exist_ok=True)

			# crate chromadb 
			self.client = chromadb.PersistentClient(path=self.persist_directory)

			# make a collection
			self.collection = self.client.get_or_create_collection(
				name = self.collection_name,
				metadata={"description":"PDF document embeddings for RAG2"}
			)
			print(f"Vector store successfully initialized.\nCollection: {self.collection_name}")
			print(f"Existing documents in collection: {self.collection.count()}")
		except Exception as e:
			print(f"Error initializing vector database: {e}")
			raise
	
	def add_documents(self, documents: List[Any], embeddings: np.ndarray):
		if len(documents) != len(embeddings):
			raise ValueError("Number of documents must match number of embeddings")
		
		print(f"Adding {len(documents)} documents to vector store...")

		ids = []
		metadatas = []
		documents_text = []
		embeddings_list = []

		for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
			doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
			ids.append(doc_id)

			# metadata
			metadata = dict(doc.metadata)
			metadata['doc_index'] = i
			metadata['content_length'] = len(doc.page_content)
			metadatas.append(metadata)
			documents_text.append(doc.page_content)
			embeddings_list.append(embedding.tolist())

		try:
			self.collection.add(
				ids = ids,
				embeddings = embeddings_list,
				metadatas=metadatas,
				documents=documents_text
			)
			print(f"succesfully added {len(documents)} documents to vector store")
			print(f"Total documents in collection: {self.collection.count()}")

		except Exception as e:
			print(f"Error adding documents to vector store: {e}")
			raise

In [32]:
vectore_store = VectorStore()

Vector store successfully initialized.
Collection: pdf_documents
Existing documents in collection: 0


In [33]:
embeddings = embedding_manager.generate_embdeddings(texts)
vectore_store.add_documents(chunks, embeddings)

Generating embeddings for 350 texts...


Batches: 100%|██████████| 11/11 [00:01<00:00,  8.86it/s]


Genrerated embeddings with shape: (350, 384)
Adding 350 documents to vector store...
succesfully added 350 documents to vector store
Total documents in collection: 350


In [46]:
# retriever pipeline

class RAGRetriever:

	def __init__(self, vector_store:VectorStore,
			  embedding_manager:EmbeddingManager):
		
		self.vector_store = vector_store
		self.embedding_manager = embedding_manager

	def retrieve(self, query:str, top_k=5, score_threshold:float=0.0) -> List[Dict[str, Any]]:
		print(f"Retrieving documents from query: '{query}'")
		print(f"Top K: {top_k}, score threshold: {score_threshold}")

		query_embedding = self.embedding_manager.generate_embdeddings([query])[0]

		try:
			results = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()],n_results=top_k)

			retrieved_docs = []

			if results['documents'] and results['documents'][0]:
				documents = results['documents'][0]
				metadatas = results['metadatas'][0]
				distances = results['distances'][0]
				ids = results['ids'][0]

				for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
					similarity_score = 1-distance
					
					if similarity_score>=score_threshold:
						retrieved_docs.append({
							'id':doc_id,
							'content':document,
							'metadata':metadata,
							'similarity_score': similarity_score,
							'distance':distance,
							'rank':i+1
						})
				print(f"Retrieved {len(retrieved_docs)} documents(after filtering)")
			else:
				print("No documents found")
			return retrieved_docs
		
		except Exception as e:
			print(f"Error during retrieval: {e}")
			raise
			return []
		

In [47]:
rag_retriever = RAGRetriever(vectore_store, embedding_manager)

In [49]:
rag_retriever.retrieve('What is the climate like in Phuket?')

Retrieving documents from query: 'What is the climate like in Phuket?'
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s]

Genrerated embeddings with shape: (1, 384)
Retrieved 5 documents(after filtering)


[{'id': 'doc_b6141a4f_194',
  'content': 'the memorial statue of the heroines Thao \nThep Krasattri and Thao Sisunthon, who rallied  \nthe islanders in 1785 to repel Burmese invaders.\nBlessed by nature and the smiling hospitality \nof its people, and supported by superb tourism \nfacilities, Phuket is today one of the world’s \npremier tropical resorts. Palm-fringed beaches, \nan island-studded sea, superb accommodation, \ndelicious seafood, numerous sporting and  \nleisure opportunities and, of course, year-round \nsunshine, are a few of what makes a trip to \nPhuket a truly memorable holiday.  \nCLIMATE\nPhuket has two main seasons: rainy from  \nMay through to October and hot from  \nNovember to April. However, there are sunny \ndays throughout the wet season; showers \ncustomarily lasting little more than 2-3 hours. \nThe best months to visit are November to April. \nAverage temperatures range between 22 and \n34 degrees Celsius. \nBEACHES & BAYS\nPhuket’s glory is its magnificent

In [51]:
# pass to llm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API")
llm = ChatGroq(groq_api_key =groq_api_key,
			   model='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
	results = retriever.retrieve(query, top_k=top_k)
	context = "\n\n".join([doc['content'] for doc in results]) if results else ""

	if not context:
		return "No relevant context found to answer the question"
	
	prompt=f"""use the following context to answer the question clearly.
	Context:
	{context} 
	Question:
	{query}

	Answer:
"""
	response = llm.invoke([prompt.format(context=context, query=query)])
	return response.content

In [55]:
answer = rag_simple("What are some activities to do in Phuket", rag_retriever, llm)
print(answer)

Retrieving documents from query: 'What are some activities to do in Phuket'
Top K: 3, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.23it/s]

Genrerated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering)


Based on the provided context, some activities to do in Phuket include:

1. **Beach activities**: Phuket is known for its beautiful beaches, so you can enjoy swimming, sunbathing, and other water sports.
2. **Major Sights & Attractions**: Phuket has many attractions such as temples, museums, and historical sites that you can visit.
3. **Recreations**: This section is not specified in the context, but it's likely to include activities like hiking, trekking, or other outdoor pursuits.
4. **Special Events**: Phuket hosts various events throughout the year, such as festivals, concerts, and cultural performances.
5. **Water sports**: Phuket's beaches offer opportunities for snorkeling, diving, kayaking, and other water sports.

Some specific activities mentioned in the context include:

- Beaches & Bays (page 9)
- Major Sights & Attractions (page 11)
- Recreations (page 15)
- Special Events (page 18)

Please note that the context is limited, and there might be more activities to do in Phuke